In [ ]:
!pip install torch_geometric

In [ ]:
!pip install igraph

#Graph Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import gc
import torch
import torch.nn.functional as F
import numpy as np
import random
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from igraph import Graph

/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: libcudart.so.11.0: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [ ]:
GRAPH_PATH = "/content/drive/MyDrive/I -TEAM FILES/Datasets/Comment_Struc (Features) - Graph.graphml"
g = Graph.Read_GraphML(GRAPH_PATH)
print(f"Loaded graph with {g.vcount()} nodes and {g.ecount()} edges")

Loaded graph with 325320 nodes and 6112052 edges


<ipython-input-5-1bab8427e92a>:2: RuntimeWarning: Could not add vertex ids, there is already an 'id' vertex attribute. at src/io/graphml.c:488
  g = Graph.Read_GraphML(GRAPH_PATH)


In [ ]:
edges = g.get_edgelist()
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [ ]:
exclude_attrs = {"id", "label"}
feature_keys = [k for k in g.vs.attributes() if k not in exclude_attrs]

In [ ]:
x = torch.tensor([
    [v[attr] if v[attr] is not None else 0.0 for attr in feature_keys]
    for v in g.vs
], dtype=torch.float)

In [ ]:
labels_raw = [v["label"] for v in g.vs]

labeled_indices = [i for i, lbl in enumerate(labels_raw) if lbl not in [None, "None"]]
labeled_labels = [labels_raw[i] for i in labeled_indices]

#real labels
label_encoder = LabelEncoder()
encoded_known_labels = label_encoder.fit_transform(labeled_labels)

y = torch.full((len(labels_raw),), -1, dtype=torch.long)
for i, enc in zip(labeled_indices, encoded_known_labels):
    y[i] = enc

In [ ]:
# training mask of labeled nodes
train_mask = y != -1

In [ ]:
print("Encoded classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

Encoded classes: ['Cyborg' 'Human' 'Spam Bot' 'Troll']
Number of classes: 4


In [ ]:
data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = train_mask

In [ ]:
print(f"Feature matrix shape: {x.shape}")
print(f"Labeled nodes: {train_mask.sum().item()} out of {train_mask.shape[0]}")
print(f"Graph: {data}")

Feature matrix shape: torch.Size([325320, 16])
Labeled nodes: 840 out of 325320
Graph: Data(x=[325320, 16], edge_index=[2, 6112052], y=[325320], train_mask=[325320])


In [ ]:
print(g.vs.attributes())

['id', 'label', 'averageCommentLength', 'averageLinkUsage', 'averageResponseTime', 'engagement', 'averagePairwiseCommentDisimilarity', 'threadDeviation', 'innovationRate', 'averageLikesPerComment', 'replyRatio', 'commentTimeVariance', 'uniquePostsCommentedOn', 'uniqueNewsOutletsCommentedOn', 'averageCommentsPerPost', 'duplicateCommentRate', 'averageMediaUsage', 'numOfComments']


# GAT Model

In [ ]:
# device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

In [ ]:
# class weights
valid_labels = y[train_mask].cpu().numpy()
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(valid_labels), y=valid_labels)
weights = torch.tensor(class_weights, dtype=torch.float).to(device)
num_classes = len(label_encoder.classes_)
target_names = label_encoder.classes_.tolist()

In [ ]:
# reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=2, dropout_rate=0.3):
        super(GAT, self).__init__()
        self.dropout_rate = dropout_rate
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, concat=True)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
# training and evaluation
def train_and_evaluate(train_mask, val_mask, test_mask, fold):
    model = GAT(
        in_channels=data.num_features,
        hidden_channels=64,
        out_channels=num_classes,
        heads=2,
        dropout_rate=0.3
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss(weight=weights)

    for epoch in range(1, 101):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out[train_mask], data.y[train_mask])
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                pred = out.argmax(dim=1)
                val_acc = (pred[val_mask] == data.y[val_mask]).float().mean().item()
                print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}")

    model.eval()
    with torch.no_grad():
        out = model(data)
        preds = out[test_mask].argmax(dim=1).cpu().numpy()
        labels = data.y[test_mask].cpu().numpy()

        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='weighted')
        cm = confusion_matrix(labels, preds)

        print(f"\nFold {fold + 1} Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
        disp.plot(cmap='Blues')
        plt.title(f"Confusion Matrix - Fold {fold + 1}")
        plt.show()
        plt.close()

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return preds, labels

In [ ]:
# K-Fold Cross-Validation
valid_nodes = (data.y != -1).nonzero(as_tuple=True)[0]
y_valid = data.y[valid_nodes].cpu().numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_preds_total, all_labels_total = [], []

for fold, (train_full_idx, test_idx) in enumerate(skf.split(valid_nodes, y_valid)):
    print(f"\nFold {fold + 1}/5")

    train_idx, val_idx = train_test_split(
        train_full_idx, test_size=0.2, random_state=42, stratify=y_valid[train_full_idx]
    )

    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

    train_mask[valid_nodes[train_idx]] = True
    val_mask[valid_nodes[val_idx]] = True
    test_mask[valid_nodes[test_idx]] = True

    preds_fold, labels_fold = train_and_evaluate(train_mask, val_mask, test_mask, fold)
    all_preds_total.extend(preds_fold)
    all_labels_total.extend(labels_fold)

In [ ]:
# final report
print("\nFinal Classification Report")
print(classification_report(all_labels_total, all_preds_total, target_names=target_names))
print(f"Overall Accuracy: {accuracy_score(all_labels_total, all_preds_total):.4f}")

cm = confusion_matrix(all_labels_total, all_preds_total)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues')
plt.title("Final Confusion Matrix")
plt.show()

## Hypertuning

In [ ]:
# # number of classes
# num_classes = len(torch.unique(data.y[data.y != -1]))

In [ ]:
# # search space
# search_space = {
#     "num_heads": [1, 2, 4, 8],
#     "hidden_dim": [8, 16, 32, 64],
#     "dropout": [0.3, 0.5, 0.6],
#     "learning_rate": [1e-3, 5e-3, 1e-2],
#     "weight_decay": [0, 5e-4, 1e-3]
# }

In [ ]:
# # random combinations
# combinations = list(product(*search_space.values()))
# random.shuffle(combinations)

# best_val_acc = 0
# best_config = None

# # Filter valid indices (no label -1)
# valid_train_idx = train_idx[data.y[train_idx] != -1]
# valid_val_idx = val_idx[data.y[val_idx] != -1]

# for trial, params in enumerate(combinations[:20]):  # Try 20 configs
#     heads, hdim, drop, lr, wd = params

#     model = GAT(
#         in_channels=data.num_node_features,
#         hidden_channels=hdim,
#         out_channels=num_classes,
#         heads=heads,
#         dropout_rate=drop
#     ).to(device)

#     optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

#     valid_labels = data.y[data.y != -1].cpu().numpy()
#     class_weights = compute_class_weight(
#         class_weight='balanced',
#         classes=np.unique(valid_labels),
#         y=valid_labels
#     )
#     weights = torch.tensor(class_weights, dtype=torch.float).to(device)
#     criterion = torch.nn.CrossEntropyLoss(weight=weights)

#     best_trial_acc = 0
#     for epoch in range(1, 101):  # Shorter loop for tuning
#         model.train()
#         optimizer.zero_grad()
#         out = model(data)
#         loss = criterion(out[valid_train_idx], data.y[valid_train_idx])
#         loss.backward()
#         optimizer.step()

#         model.eval()
#         pred = out.argmax(dim=1)
#         val_correct = (pred[valid_val_idx] == data.y[valid_val_idx]).sum().item()
#         val_acc = val_correct / len(valid_val_idx)

#         if val_acc > best_trial_acc:
#             best_trial_acc = val_acc

#     print(f"Trial {trial}: Val Acc = {best_trial_acc:.4f} with config {params}")

#     if best_trial_acc > best_val_acc:
#         best_val_acc = best_trial_acc
#         best_config = params

# print(f"Best Config: {best_config} with Val Acc = {best_val_acc:.4f}")

##**For test**

###**LOADING FULL ANNOTATIONS**

In [ ]:
import json

with open("/content/drive/MyDrive/I -TEAM FILES/Datasets/Facebook_Comments (Fully Annotated).json", "r") as f:
    annotations = json.load(f)

In [ ]:
user_label_map = {entry["userID"]: entry["label"] for entry in annotations}

###**LOADING GRAPH**

In [ ]:
GRAPH_PATH = "/content/drive/MyDrive/I -TEAM FILES/Datasets/Comment_Struc (Features) - Graph.graphml"

In [ ]:
try:
    import igraph
except ImportError:
    print("Installing igraph...")
    !pip install igraph


from igraph import Graph

# Load the graph
g = Graph.Read_GraphML(GRAPH_PATH)
print(f"Loaded graph with {g.vcount()} nodes and {g.ecount()} edges")


Installing igraph...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.0 MB/s eta 0:00:00


<ipython-input-5-6b5f6466bd92>:11: RuntimeWarning: Could not add vertex ids, there is already an 'id' vertex attribute. at src/io/graphml.c:488
  g = Graph.Read_GraphML(GRAPH_PATH)


Loaded graph with 325320 nodes and 6112052 edges


In [ ]:
for v in g.vs:
    uid = v["id"]
    v["label"] = user_label_map.get(uid, None)

In [ ]:
try:
    import torch_geometric
except ImportError:
    print("Installing torch_geometric...")
    !pip install torch_geometric

import torch
from torch_geometric.data import Data

# 1. Build edge index
edges = g.get_edgelist()
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()  # shape: [2, num_edges]

Installing torch_geometric...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.9 MB/s eta 0:00:00


In [ ]:
exclude_attrs = {"id", "label"}
feature_keys = [k for k in g.vs.attributes() if k not in exclude_attrs]

In [ ]:
x = torch.tensor([
    [v[attr] if v[attr] is not None else 0.0 for attr in feature_keys]
    for v in g.vs
], dtype=torch.float)

In [ ]:
# Clean label list properly
labels_raw = [v["label"] for v in g.vs]

# Accept only real labels, excluding NoneType and string "None"
labeled_indices = [i for i, lbl in enumerate(labels_raw) if lbl not in [None, "None"]]
labeled_labels = [labels_raw[i] for i in labeled_indices]

# Encode only real labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
encoded_known_labels = label_encoder.fit_transform(labeled_labels)

# Create label tensor initialized with -1 (unlabeled)
y = torch.full((len(labels_raw),), -1, dtype=torch.long)
for i, enc in zip(labeled_indices, encoded_known_labels):
    y[i] = enc

# Mask of labeled nodes
train_mask = y != -1

In [ ]:
print("Encoded classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))  # Should now be 4


Encoded classes: ['Cyborg' 'Human' 'Spam Bot' 'Troll']
Number of classes: 4


In [ ]:
test_mask = torch.zeros_like(y, dtype=torch.bool)  # No separate test set for now
data = Data(x=x, edge_index=edge_index, y=y)

In [ ]:
print(f"Feature matrix shape: {x.shape}")
print(f"Number of classes: {len(label_encoder.classes_)}")
print(f"Labeled nodes: {train_mask.sum().item()} out of {train_mask.shape[0]}")
print(f"Graph: {data}")

Feature matrix shape: torch.Size([325320, 16])
Number of classes: 4
Labeled nodes: 325320 out of 325320
Graph: Data(x=[325320, 16], edge_index=[2, 6112052], y=[325320])


###**GAT MODEL**

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import torch.nn as nn
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os
import psutil
from tqdm.notebook import tqdm


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# GAT Model Definition
class GAT(torch.nn.Module):
    def __init__(self, in_channels, out_channels, num_heads=1, hidden_dim=8):
        super(GAT, self).__init__()

        # Single hidden layer with smaller dimensions for Colab
        self.gat1 = GATConv(in_channels, hidden_dim, heads=num_heads, dropout=0.3)
        self.gat2 = GATConv(hidden_dim * num_heads, out_channels, heads=1, dropout=0.3)

        # Batch normalization for faster convergence
        self.bn = torch.nn.BatchNorm1d(hidden_dim * num_heads)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index):
        # Apply first layer with activation
        x = self.gat1(x, edge_index)

        # Apply normalization (handles NaN/Inf values better)
        x = self.bn(x)
        x = F.elu(x)  # ELU is often more stable than ReLU
        x = self.dropout(x)

        # Output layer
        x = self.gat2(x, edge_index)

        return x

In [ ]:
# Training Function
def train(model, data, optimizer, criterion):
  model.train()
  optimizer.zero_grad()

  # Forward pass with catching potential errors
  try:
      # Forward pass
      out = model(data.x, data.edge_index)

      # Check for NaN outputs and handle
      if torch.isnan(out).any():
          print("Warning: NaN values detected in model output, attempting recovery...")
          return 999.0  # Return high loss to trigger scheduler

      loss = criterion(out[data.train_mask], data.y[data.train_mask])

      # Backward pass with gradient clipping
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()

      return loss.item()

  except RuntimeError as e:
      if "out of memory" in str(e):
          print("WARNING: GPU out of memory, attempting recovery...")
          torch.cuda.empty_cache()
          return 999.0  # Return high loss to signal problems
      else:
          raise e

In [ ]:
def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        try:
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)

            true_labels = data.y[data.val_mask].cpu()
            val_pred = pred[data.val_mask].cpu()

            accuracy = accuracy_score(true_labels, val_pred)
            precision = precision_score(true_labels, val_pred, average='macro', zero_division=1)
            recall = recall_score(true_labels, val_pred, average='macro', zero_division=1)
            f1 = f1_score(true_labels, val_pred, average='macro', zero_division=1)

            # AUC-ROC can only be computed for binary or multi-label with probabilities
            try:
                probs = F.softmax(out[data.val_mask], dim=1).cpu()
                auc_roc = roc_auc_score(true_labels, probs, multi_class='ovo')
            except ValueError:
                auc_roc = float('nan')

            return accuracy, precision, recall, f1, auc_roc, true_labels, val_pred

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("WARNING: GPU OOM during evaluation, attempting recovery...")
                torch.cuda.empty_cache()
                return 0.0, 0.0, 0.0, 0.0, 0.0, None, None
            else:
                raise e

In [ ]:
labeled_indices = torch.where(y != -1)[0].numpy()
kf = KFold(n_splits=10, shuffle=True, random_state=42)

#base data object
base_data = Data(x=x, edge_index=edge_index, y=y)

# Store results
all_accuracy, all_precision, all_recall, all_f1, all_auc_roc = [], [], [], [], []
confusion_matrices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(labeled_indices)):
    print(f"\nFold {fold+1}/10")

    # Define masks
    train_mask = torch.zeros(y.size(0), dtype=torch.bool)
    val_mask = torch.zeros(y.size(0), dtype=torch.bool)
    train_mask[labeled_indices[train_idx]] = True
    val_mask[labeled_indices[val_idx]] = True

    # Construct data with new masks
    data = Data(
        x=base_data.x,
        edge_index=base_data.edge_index,
        y=base_data.y,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=torch.zeros_like(train_mask)  # Dummy
    )

    # Use lightweight GAT model
    model = GAT(in_channels=x.shape[1], out_channels=len(label_encoder.classes_), num_heads=2, hidden_dim=16)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(50):  # fewer epochs
        loss = train(model, data, optimizer, criterion)

    # Evaluate
    accuracy, precision, recall, f1, auc_roc, true_labels, val_pred = evaluate(model, data)
    all_accuracy.append(accuracy)
    all_precision.append(precision)
    all_recall.append(recall)
    all_f1.append(f1)
    all_auc_roc.append(auc_roc)

    # Store confusion matrix
    cm = confusion_matrix(true_labels.cpu(), val_pred.cpu())
    confusion_matrices.append(cm)

    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

    # Memory cleanup
    del model, optimizer, data
    torch.cuda.empty_cache()
    gc.collect()


Fold 1/10


In [ ]:
print("\nAverage Metrics Across 10 Folds:")
print(f"Accuracy : {np.mean(all_accuracy):.4f}")
print(f"Precision: {np.mean(all_precision):.4f}")
print(f"Recall   : {np.mean(all_recall):.4f}")
print(f"F1 Score : {np.mean(all_f1):.4f}")
print(f"AUC-ROC  : {np.mean(all_auc_roc):.4f}")


In [ ]:
for i, cm in enumerate(confusion_matrices):
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f"Confusion Matrix - Fold {i+1}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()
    plt.close()